In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('cars_dataset.csv')

In [4]:
df.shape

(8000, 5)

In [5]:
# for making some missing values 
np.random.seed(42)
missing_km_indices = np.random.choice(df.index, size = int(0.05*len(df)), replace=False)
df.loc[missing_km_indices, 'km_driven'] = np.nan

#Introduce missing value in 'owner' column (1% missing values)
missing_owner_indices = np.random.choice(df.index, size = int(0.01*len(df)), replace=False)
df.loc[missing_owner_indices, 'owner'] = np.nan

In [6]:
df.isnull().sum()

brand              0
km_driven        400
fuel               0
owner             80
selling_price      0
dtype: int64

In [7]:
# plan of Attack

# Missing values Imputation
# Encoding categorical variables 
# Scaling 
# Feature Selection
# Model Building - RF
# Predictions

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
                                                    df.drop(columns=['selling_price']),
                                                    df['selling_price'], 
                                                    test_size=0.2, 
                                                    random_state=42
                                                )

In [12]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Skoda,133038.0,Diesel,Second Owner,593000
1,Maruti,194721.0,Petrol,First Owner,694000
2,Toyota,14213.0,Diesel,Second Owner,458000
3,Tata,69210.0,Diesel,First Owner,1134000
4,Ford,10406.0,Diesel,First Owner,750000


In [ ]:
# Imputation transformers - 
# SimpleImputer() - Which is the process of replacing missing values with means(like NaN or None) in a dataset with estimated values. 

trf1 = ColumnTransformer([
      ('impute_km_driven', SimpleImputer(), [1]),
      ('impute_owner', SimpleImputer(strategy='most_frequent'), [3]) # 'most_frequent' it means ki fill the missing values with most common values 
], remainder='passthrough')

In [ ]:
# encoding categorical variables 

trf2 = ColumnTransformer([
      ('Ordinal', OrdinalEncoder(), [3]),
      ('onehot',  OneHotEncoder(sparse_output=False), [0,2])
], remainder='passthrough')

In [22]:
# scaling - one all col(0, 38)
trf3 = ColumnTransformer([
      ('scale', MinMaxScaler(), slice(0,38))
])

In [ ]:
# Feature selection - to find 10 best col.
trf4 = SelectKBest(score_func=chi2, k=10)

In [20]:
# train the model 
trf5 = RandomForestRegressor()

In [ ]:
# So, here will be the independent tranformers, so in next step we well connect these 5 transformer with each other.

In [24]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
      ('impute', trf1),
      ('encoder', trf2),
      ('scaler', trf3),
      ('fselection', trf4),
      ('model', trf5)
])

# flow to connect
# pipe = Pipeline([(), (), ()]) -  each tuple is transformer

In [27]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('impute',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_km_driven',
                                                  SimpleImputer(), [1]),
                                                 ('impute_owner',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [3])])),
                ('encoder',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Ordinal', OrdinalEncoder(),
                                                  [3]),
                                                 ('onehot',
                                                  OneHotEncoder(sparse_output=False),
                                                  [0, 2])])),
                ('scaler',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 38, None))])),
                ('fselection',
                 SelectKBest(score_func=<function chi2 at 0x000002024CD2E340>)),
                ('model', RandomForestRegressor())])

In [28]:
pipe.feature_names_in_

array(['brand', 'km_driven', 'fuel', 'owner'], dtype=object)

In [46]:
pipe.named_steps

{'impute': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_km_driven', SimpleImputer(), [1]),
                                 ('impute_owner',
                                  SimpleImputer(strategy='most_frequent'),
                                  [3])]),
 'encoder': ColumnTransformer(remainder='passthrough',
                   transformers=[('Ordinal', OrdinalEncoder(), [3]),
                                 ('onehot', OneHotEncoder(sparse_output=False),
                                  [0, 2])]),
 'scaler': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 38, None))]),
 'fselection': SelectKBest(score_func=<function chi2 at 0x000002024CD2E340>),
 'model': RandomForestRegressor()}

In [47]:
# to find the means value - time 1:20:00
print(f"Means: {pipe.named_steps['impute'].transformers_[0][1].statistics_}" )
print(f"Most frequent: {pipe.named_steps['impute'].transformers_[1][1].statistics_ }")

Means: [156016.4075]
Most frequent: ['First Owner']


In [ ]:
# MinMaxScaler - result 

print(f"Data Min: {pipe.named_steps['scaler'].transformers_[0][1].data_min_}\n")
print(f"Data Max: {pipe.named_steps['scaler'].transformers_[0][1].data_max_}")

Data Min: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Data Max: [3. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
